
# Using RetrieveChat Powered by MongoDB Atlas for Retrieval-Augmented Code Generation and Question Answering

[![AI Learning Hub For Developers](https://img.shields.io/badge/AI%20Learning%20Hub%20For%20Developers-Click%20Here-blue)](https://www.mongodb.com/resources/use-cases/artificial-intelligence?utm_campaign=ai_learning_hub&utm_source=github&utm_medium=referral)
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-developer/GenAI-Showcase/blob/main/notebooks/agents/agentchat_RetrieveChat_mongodb.ipynb)

AutoGen offers conversable agents powered by an LLM, tools, or humans, which can be used to perform tasks collaboratively through automated chat. This framework allows tool use and human participation through multi-agent conversations.
You can find documentation for this feature [here](https://microsoft.github.io/autogen/docs/Use-Cases/agent_chat).

RetrieveChat is a conversational system for retrieval-augmented code generation and question answering. In this notebook, we demonstrate how to use RetrieveChat to generate code and answer questions based on custom documentation that is not present in the LLM's training dataset. RetrieveChat uses the `AssistantAgent` and `RetrieveUserProxyAgent`, similar to the usage of `AssistantAgent` and `UserProxyAgent` in other notebooks (e.g., [Automated Task Solving with Code Generation, Execution & Debugging](https://github.com/microsoft/FLAML/blob/main/notebook/autogen_agentchat_auto_feedback_from_code_execution.ipynb)). Essentially, `RetrieveUserProxyAgent` implements a different auto-reply mechanism corresponding to the RetrieveChat prompts.

## Requirements

Ensure you have a MongoDB Atlas instance with cluster tier >= M10. Read more about cluster support [here](https://www.mongodb.com/docs/atlas/atlas-search/manage-indexes/#create-and-manage-fts-indexes).

After you deploy your MongoDB Atlas instance, get the connection string and set it either as an environment variable or when prompted in the next cell.

Additionally, for this notebook, you will need to provide an OpenAI API key. Same as the MongoDB connection string, you can set it as an environment variable or provide it when prompted in the next cell.

In [1]:
import os
from getpass import getpass
from dotenv import load_dotenv

load_dotenv()

def get_or_prompt_env(var_name: str, prompt_text: str) -> str:
    value = os.environ.get(var_name)
    if value:
        return value

    value = getpass(prompt_text)
    if not value:
        raise EnvironmentError(f"Environment variable {var_name} is required.")

    os.environ[var_name] = value
    return value


OPENAI_API_KEY = get_or_prompt_env("OPENAI_API_KEY", "Enter OPENAI_API_KEY: ")
MONGODB_URI = get_or_prompt_env("MONGODB_URI", "Enter MONGODB_URI: ")

print("Environment variables loaded successfully")

Environment variables loaded successfully


Let's test that the MongoDB connection string is working. We'll do that by pinging the MongoDB Atlas instance and checking if we can connect to it. If the connection is successful, we will proceed with the rest of the notebook.

In [2]:
import pymongo

mongodb_client = pymongo.MongoClient(MONGODB_URI)
try:
    # The ping command is cheap and does not require auth.
    mongodb_client.admin.command('ping')
    print("Pinged your deployment. You successfully connected to MongoDB!")
except Exception as e:
    print(f"Failed to connect to MongoDB: {e}")

Pinged your deployment. You successfully connected to MongoDB!


Next, we'll test that the OpenAI API key is working. We'll do that by making a simple request to the OpenAI API and checking if we get a valid response. If the request is successful, we will proceed with the rest of the notebook.

In [3]:
import openai

openai_client = openai.OpenAI(api_key=OPENAI_API_KEY)
try:
    # Make a simple request to the OpenAI API to test the API key
    response = openai_client.models.list()
    print("Successfully connected to OpenAI API!")
except Exception as e:
    print(f"Failed to connect to OpenAI API: {e}")

Successfully connected to OpenAI API!


Then, install the required packages.

In [ ]:
%pip install -U -q 'ag2[retrievechat-mongodb]' 'flaml[automl]'

## Set your API credentials


In [ ]:
from autogen import AssistantAgent
from autogen.agentchat.contrib.retrieve_user_proxy_agent import RetrieveUserProxyAgent
from autogen.retrieve_utils import TEXT_FORMATS

# Accepted file formats that can be stored in a vector database instance.

config_list = [
    {
        "model": "gpt-3.5-turbo",
        "api_key": OPENAI_API_KEY,
        "api_type": "openai",
    }
]
print("Models to use:", [item["model"] for item in config_list])

In [6]:
print("Accepted file formats for `docs_path`:")
print(TEXT_FORMATS)

Accepted file formats for `docs_path`:
['txt', 'json', 'csv', 'tsv', 'md', 'html', 'htm', 'rtf', 'rst', 'jsonl', 'log', 'xml', 'yaml', 'yml', 'pdf', 'mdx']


In [7]:
# 1. Create an AssistantAgent instance named "assistant".
assistant = AssistantAgent(
    name="assistant",
    system_message="You are a helpful assistant.",
    llm_config={
        "timeout": 600,
        "cache_seed": 42,
        "config_list": config_list,
    },
)

# 2. Create the RetrieveUserProxyAgent instance named "ragproxyagent".
# Refer to https://microsoft.github.io/autogen/docs/reference/agentchat/contrib/retrieve_user_proxy_agent
# and https://microsoft.github.io/autogen/docs/reference/agentchat/contrib/vectordb/mongodb
# for more information on RetrieveUserProxyAgent and MongoDBAtlasVectorDB.
ragproxyagent = RetrieveUserProxyAgent(
    name="ragproxyagent",
    human_input_mode="NEVER",
    max_consecutive_auto_reply=3,
    retrieve_config={
        "task": "code",
        "docs_path": [
            "https://raw.githubusercontent.com/microsoft/FLAML/main/website/docs/Examples/Integrate%20-%20Spark.md",
            "https://raw.githubusercontent.com/microsoft/FLAML/main/website/docs/Research.md",
        ],
        "chunk_token_size": 2000,
        "model": config_list[0]["model"],
        "vector_db": "mongodb",  # MongoDB Atlas database
        "collection_name": "demo_collection",
        "db_config": {
            "connection_string": MONGODB_URI,  # MongoDB Atlas connection string
            "database_name": "test_db",
            "index_name": "vector_index",
            "wait_until_index_ready": 120.0,
            "wait_until_document_ready": 120.0,
        },
        "get_or_create": True,  # Set to False if you do not want to reuse an existing collection.
        "overwrite": False,  # Set to True if you want to overwrite an existing collection.
    },
    code_execution_config=False,  # Set to False if you do not want to execute generated code.
    # For AG2 >= 0.8.0 use code_execution_config={"use_docker": False}.
    # Keep this bool form for backward compatibility with older releases.
)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 8920.00it/s]


### Code generation with RetrieveChat

Use RetrieveChat to generate sample code, run it automatically, and fix errors if any occur.

Problem: Which API should I use if I want to use FLAML for a classification task and train the model in 30 seconds? Use Spark to parallelize training. Force-cancel jobs if the time limit is reached.

In [8]:
# Reset the assistant. Always reset the assistant before starting a new conversation.
assistant.reset()

# Given a problem, we use ragproxyagent to generate a prompt for the assistant.
# The assistant receives the message and generates a response.
# The response is sent back to ragproxyagent for processing.
# The conversation continues until the termination condition is met.
# In RetrieveChat without human-in-the-loop, termination occurs when no code block is detected.
# With human-in-the-loop, the conversation continues until the user says "exit".
code_problem = (
    "How can I use FLAML to perform a classification task and use Spark "
    "for parallel training? Train for 30 seconds and force-cancel jobs "
    "if the time limit is reached."
)
chat_result = ragproxyagent.initiate_chat(
    assistant, message=ragproxyagent.message_generator, problem=code_problem
)

2026-07-08 17:44:31,776 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Use the existing collection `demo_collection`.


Trying to create collection.


2026-07-08 17:44:35,147 - autogen.agentchat.contrib.retrieve_user_proxy_agent - INFO - Found 2 chunks.
2026-07-08 17:44:35,222 - autogen.agentchat.contrib.vectordb.mongodb - INFO - No documents to insert.


VectorDB returns doc_ids:  [['bdfbc921', '7968cf3c']]
Adding content of doc bdfbc921 to context.
Adding content of doc 7968cf3c to context.
ragproxyagent (to assistant):

You're a retrieve augmented coding assistant. You answer user's questions based on your own knowledge and the
context provided by the user.
If you can't answer the question with or without the current context, you should reply exactly `UPDATE CONTEXT`.
For code generation, you must obey the following rules:
Rule 1. You MUST NOT install any packages because all the packages needed are already installed.
Rule 2. You must follow the formats below to write your code:
```language
# your code
```

User's question is: How can I use FLAML to perform a classification task and use Spark for parallel training? Train for 30 seconds and force-cancel jobs if the time limit is reached.

Context is: # Integrate - Spark

FLAML has integrated Spark for distributed training. There are two main aspects of integration with Spark:

- Use S

assistant (to ragproxyagent):

To use FLAML to perform a classification task and use Spark for parallel training with a time limit of 30 seconds and force-cancel jobs if the time limit is reached, you can follow these steps:

```python
import flaml

# Prepare your data in pandas-on-spark format as previously mentioned
# Make sure you have your data loaded into a pandas-on-spark dataframe psdf and specify the label

automl = flaml.AutoML()
settings = {
    "time_budget": 30,
    "metric": "accuracy",  # Use the appropriate metric for classification
    "estimator_list": ["lgbm_spark"],  # This setting is optional
    "task": "classification",
    "n_concurrent_trials": 2,
    "use_spark": True,
    "force_cancel": True,  # Activating the force_cancel option can immediately halt Spark jobs if the time budget is exceeded.
}

automl.fit(
    dataframe=psdf,
    label=label,
    **settings,
)
```

This code snippet demonstrates how to configure FLAML to perform a classification task using S

assistant (to ragproxyagent):

```UPDATE CONTEXT```

--------------------------------------------------------------------------------
Updating context and resetting conversation.
VectorDB returns doc_ids:  [['bdfbc921', '7968cf3c']]
No more context, will terminate.
ragproxyagent (to assistant):

TERMINATE

--------------------------------------------------------------------------------

>>>>>>>> TERMINATING RUN (883bde6b-3870-494f-83e8-55168699b471): Termination message condition on agent 'assistant' met
